# 14 — Manual interactive 3D skull/brain multiplanar dashboard

This notebook contains **one coordinated dashboard**, not separate viewers.

Dashboard layout:

- **Left:** rotatable patient-specific 3D reconstruction.
- **Right:** separate sagittal, coronal and transverse/axial CT outputs.
- **Top model selector:** Skull, Brain, Skull + Brain, or Neural CT Brain.
- **Plane checkboxes:** show any one plane, any two planes, or all three.
- **Manual mouse sliders:** moving a slider moves the transparent glass plane in the 3D model and immediately updates its matching 2D CT output.
- **CSF display:** optional cleaned native-CT 0–20 HU surface with user-selectable color and opacity. The original unsmoothed mask remains unchanged for measurements and the 2D contour.
- **Combined-view opacity:** separate skull and brain opacity sliders appear only in the Skull + Brain mode.

There is no automatic playback or animation. The sliders respond only to the user.

In [1]:
from pathlib import Path
import base64, json, warnings

import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to
from scipy import ndimage as ndi
from skimage.measure import marching_cubes
from plotly.offline import get_plotlyjs
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path(r"F:\csstdy")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "14_interactive_multiplanar_3d"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CT_PATH = PROJECT_ROOT / "outputs/09_totalsegmentator_dl_skull_brain/input/CASE_001_brain_kernel_ct.nii.gz"
BRAIN_PATH = PROJECT_ROOT / "outputs/09_totalsegmentator_dl_skull_brain/segmentations/brain.nii.gz"
SKULL_PATH = PROJECT_ROOT / "outputs/10_totalsegmentator_sharp_skull/craniofacial_segmentations/skull.nii.gz"
NEURAL_HU_PATH = PROJECT_ROOT / "outputs/12_siren_neural_skull_brain/volumes/CASE_001_siren_soft_hu.nii.gz"
NEURAL_BRAIN_PATH = PROJECT_ROOT / "outputs/12_siren_neural_skull_brain/volumes/CASE_001_siren_brain_mask.nii.gz"
LOW_HU_SUMMARY = PROJECT_ROOT / "outputs/11_neural_ct_rendering/low_hu_verification/low_hu_verification_summary.json"

for p in [CT_PATH,BRAIN_PATH,SKULL_PATH,NEURAL_HU_PATH,NEURAL_BRAIN_PATH]:
    if not p.exists(): raise FileNotFoundError(p)

ct_img=nib.load(CT_PATH); affine=ct_img.affine.copy()
ct=ct_img.get_fdata(dtype=np.float32).copy()
brain=nib.load(BRAIN_PATH).get_fdata(dtype=np.float32)>.5
skull=nib.load(SKULL_PATH).get_fdata(dtype=np.float32)>.5

neural_img=nib.load(NEURAL_HU_PATH); neural_affine=neural_img.affine.copy()
neural_hu=neural_img.get_fdata(dtype=np.float32).copy()
neural_brain=nib.load(NEURAL_BRAIN_PATH).get_fdata(dtype=np.float32)>.5

assert ct.shape==brain.shape==skull.shape
assert neural_hu.shape==neural_brain.shape
print("Native CT:",ct.shape,nib.aff2axcodes(affine))
print("Neural CT:",neural_hu.shape,nib.aff2axcodes(neural_affine))

Native CT: (512, 512, 227) ('L', 'A', 'S')
Neural CT: (264, 326, 225) ('L', 'A', 'S')


## 1. Double-check and reconstruct the blue CSF-like candidate overlay

The blue regions are not invented annotations. They are independently recomputed from the **original CT** using the nominal analysis from notebook 11:

1. erode the TotalSegmentator brain mask by three voxels;
2. retain original-CT voxels from 0 to 20 HU;
3. use 26-connected 3D components;
4. retain components with physical volume of at least 10 mm³.

These are **CSF-like low-HU candidates**, not a definite ventricular segmentation or diagnosis.

In [2]:
inner_brain=ndi.binary_erosion(brain,iterations=3)
csf_raw=inner_brain & (ct>=0) & (ct<=20)
labels,n_components=ndi.label(csf_raw,structure=np.ones((3,3,3),dtype=np.uint8))
voxel_volume_mm3=float(abs(np.linalg.det(affine[:3,:3])))
counts=np.bincount(labels.ravel())
kept_ids=np.where(counts*voxel_volume_mm3>=10.0)[0]
kept_ids=kept_ids[kept_ids>0]
csf_candidate=np.isin(labels,kept_ids)

# Display-only cleanup: same native CT evidence, but isolated voxel noise is
# removed and only the largest persistent components are surfaced in 3-D.
csf_opened=ndi.binary_opening(csf_candidate,iterations=1)
display_labels,display_component_count=ndi.label(csf_opened,structure=np.ones((3,3,3),dtype=np.uint8))
display_counts=np.bincount(display_labels.ravel())
display_ids=np.where(display_counts>=60)[0]
display_ids=display_ids[display_ids>0]
if len(display_ids)>20:
    display_ids=display_ids[np.argsort(display_counts[display_ids])[-20:]]
csf_display_clean=np.isin(display_labels,display_ids)

reference_match=None
if LOW_HU_SUMMARY.exists():
    reference=json.loads(LOW_HU_SUMMARY.read_text(encoding="utf-8"))
    nominal=next(r for r in reference["threshold_sensitivity_results"] if r["threshold"]=="B: 0 to 20 HU")
    reference_match=(
        int(csf_raw.sum())==int(nominal["candidate_voxels"]) and
        int(n_components)==int(nominal["components_before_filtering"]) and
        len(kept_ids)==int(nominal["components_after_filtering"])
    )

csf_img=nib.Nifti1Image(csf_candidate.astype(np.uint8),affine,ct_img.header)
csf_path=OUTPUT_DIR/"CASE_001_native_CT_CSF_like_0_20HU_candidates.nii.gz"
nib.save(csf_img,csf_path)
csf_clean_path=OUTPUT_DIR/"CASE_001_native_CT_CSF_like_clean_display_mask.nii.gz"
nib.save(nib.Nifti1Image(csf_display_clean.astype(np.uint8),affine,ct_img.header),csf_clean_path)

neural_csf_img=resample_from_to(csf_img,(neural_hu.shape,neural_affine),order=0)
neural_csf=neural_csf_img.get_fdata(dtype=np.float32)>.5

print("Native 0–20 HU voxels before filtering:",int(csf_raw.sum()))
print("Connected components before filtering:",int(n_components))
print("Components retained at >=10 mm³:",len(kept_ids))
print("Retained native measurement voxels:",int(csf_candidate.sum()))
print("Clean display-surface voxels:",int(csf_display_clean.sum()),"across",len(display_ids),"components")
print("Exact match to notebook 11 nominal counts:",reference_match)
print("Saved native measurement mask:",csf_path)
print("Saved clean display mask:",csf_clean_path)

Native 0–20 HU voxels before filtering: 225958
Connected components before filtering: 2138
Components retained at >=10 mm³: 136
Retained native measurement voxels: 216138
Clean display-surface voxels: 156287 across 20 components
Exact match to notebook 11 nominal counts: True
Saved native measurement mask: F:\csstdy\outputs\14_interactive_multiplanar_3d\CASE_001_native_CT_CSF_like_0_20HU_candidates.nii.gz
Saved clean display mask: F:\csstdy\outputs\14_interactive_multiplanar_3d\CASE_001_native_CT_CSF_like_clean_display_mask.nii.gz


## 2. Prepare the 3D surfaces and compact browser data

In [3]:
def bbox(mask,pad=3):
    p=np.argwhere(mask); lo=np.maximum(p.min(0)-pad,0); hi=np.minimum(p.max(0)+pad+1,mask.shape)
    return lo.astype(int),hi.astype(int)

def mesh(mask,img_affine,sigma,step=2):
    lo,hi=bbox(mask)
    crop=mask[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(np.float32)
    crop=ndi.gaussian_filter(crop,sigma=sigma)
    v,f,_,_=marching_cubes(crop,.5,step_size=step,allow_degenerate=False)
    v+=lo[None,:]
    return nib.affines.apply_affine(img_affine,v).astype("<f4"),f.astype("<i4")

brain_v,brain_f=mesh(brain,affine,1.0)
skull_v,skull_f=mesh(skull,affine,.65)
neural_v,neural_f=mesh(neural_brain,neural_affine,.9)
csf_v,csf_f=mesh(csf_display_clean,affine,1.15,step=2)

def max_downsample(mask,steps):
    filtered=ndi.maximum_filter(mask.astype(np.uint8),size=steps,mode="constant")
    return np.ascontiguousarray(filtered[::steps[0],::steps[1],::steps[2]])

native_steps=np.array([4,4,2],dtype=int)
neural_steps=np.array([3,4,2],dtype=int)
native_ct=np.ascontiguousarray(ct[::4,::4,::2].astype("<f4"))
native_brain=max_downsample(brain,tuple(native_steps)).astype(np.uint8)
native_skull=max_downsample(skull,tuple(native_steps)).astype(np.uint8)
native_csf=max_downsample(csf_candidate,tuple(native_steps)).astype(np.uint8)

neural_ct=np.ascontiguousarray(neural_hu[::3,::4,::2].astype("<f4"))
neural_brain_ds=max_downsample(neural_brain,tuple(neural_steps)).astype(np.uint8)
neural_csf_ds=max_downsample(neural_csf,tuple(neural_steps)).astype(np.uint8)

def enc(arr,dtype):
    a=np.ascontiguousarray(arr.astype(dtype,copy=False))
    return {"b64":base64.b64encode(a.tobytes()).decode("ascii"),"shape":list(a.shape),"dtype":dtype}

def mesh_bundle(v,f): return {"v":enc(v,"<f4"),"f":enc(f,"<i4")}

bundle={
    "native":{
        "ct":enc(native_ct,"<f4"),"brain":enc(native_brain,"u1"),
        "skull":enc(native_skull,"u1"),"csf":enc(native_csf,"u1"),
        "steps":native_steps.tolist(),"affine":affine.tolist(),
    },
    "neural":{
        "ct":enc(neural_ct,"<f4"),"brain":enc(neural_brain_ds,"u1"),
        "skull":enc(np.zeros_like(neural_brain_ds),"u1"),"csf":enc(neural_csf_ds,"u1"),
        "steps":neural_steps.tolist(),"affine":neural_affine.tolist(),
    },
    "meshes":{"brain":mesh_bundle(brain_v,brain_f),"skull":mesh_bundle(skull_v,skull_f),
               "neural":mesh_bundle(neural_v,neural_f),"csf":mesh_bundle(csf_v,csf_f)},
    "csfVerification":{"rawVoxels":int(csf_raw.sum()),"componentsBefore":int(n_components),
                       "componentsAfter":int(len(kept_ids)),"referenceMatch":reference_match}
}
print("Native dashboard grid:",native_ct.shape)
print("Neural dashboard grid:",neural_ct.shape)
print("Clean 3D CSF-like surface:",len(csf_v),"vertices /",len(csf_f),"faces")

Native dashboard grid: (128, 128, 114)
Neural dashboard grid: (88, 82, 113)
Clean 3D CSF-like surface: 27780 vertices / 55380 faces


## 3. Single manual dashboard

Use the controls at the top of the dashboard. The three translucent colored rectangles inside the 3D reconstruction are only glass plane guides—CT intensity is shown in the separate right-hand panels.

The CT panels use a true grayscale window. Skull, brain and CSF-like candidates are thin contours rather than filled color layers. Separate skull/brain opacity controls appear only when **Skull + Brain** is selected.

In [4]:
DATA_JSON=json.dumps(bundle,separators=(",",":"))
PLOTLY_JS=get_plotlyjs()

HTML_TEMPLATE=r'''<!doctype html>
<html><head><meta charset="utf-8"><style>
html,body{margin:0;background:#000;color:#e6edf3;font-family:Inter,Segoe UI,Arial,sans-serif}
#app{padding:10px;box-sizing:border-box;background:#000}.top{display:flex;flex-wrap:wrap;gap:18px;align-items:center;
background:#050505;border:1px solid #242424;border-radius:9px;padding:10px 14px;margin-bottom:8px}
.group{display:flex;gap:10px;align-items:center}.group b{color:#8fd3ff}.group label{cursor:pointer;white-space:nowrap}
.note{font-size:12px;color:#9fb2c3;margin-left:auto}.grid{display:grid;grid-template-columns:minmax(590px,2fr) minmax(360px,1fr);gap:8px;height:810px}
#scene3d{min-width:0;background:#000;border:1px solid #242424;border-radius:8px}.right{display:flex;flex-direction:column;gap:6px;min-width:0;background:#000}
.panel{flex:1;min-height:0;background:#000;border:1px solid #242424;border-radius:7px;overflow:hidden}.sliders{display:grid;grid-template-columns:1fr;gap:7px;
background:#050505;border:1px solid #242424;border-radius:9px;padding:9px 14px;margin-top:8px}
.sliderrow{display:grid;grid-template-columns:150px 1fr 92px;gap:10px;align-items:center}.sliderrow input{width:100%;cursor:ew-resize}
.sag{color:#57d7ff}.cor{color:#ff68cf}.axi{color:#71e875}.blue{color:#2f9bff}.hidden{display:none!important}.opacityControl{display:none;gap:8px;align-items:center}.opacityControl input{width:105px}.opacityControl{display:none;gap:8px;align-items:center}.opacityControl input{width:105px}
@media(max-width:1000px){.grid{grid-template-columns:1fr;height:auto}#scene3d{height:650px}.right{height:900px}.note{margin-left:0}}
</style><script>__PLOTLY__</script></head><body><div id="app">
<div class="top">
 <div class="group"><b>3D model:</b>
  <label><input type="radio" name="model" value="skull"> Skull</label>
  <label><input type="radio" name="model" value="brain"> Brain</label>
  <label><input type="radio" name="model" value="combined" checked> Skull + Brain</label>
  <label><input type="radio" name="model" value="neural"> Neural CT Brain</label>
 </div>
 <div class="group"><b>Visible planes:</b>
  <label class="sag"><input id="showSag" type="checkbox" checked> Sagittal</label>
  <label class="cor"><input id="showCor" type="checkbox" checked> Coronal</label>
  <label class="axi"><input id="showAxi" type="checkbox" checked> Transverse</label>
 </div>
 <div class="group"><label class="blue"><input id="showCsf" type="checkbox"> CSF-like surface</label>
  <label>Color <input id="csfColor" type="color" value="#168bff"></label>
  <label>Opacity <input id="csfOpacity" type="range" min="0.05" max="1" step="0.05" value="0.45"></label>
 </div>
 <div class="opacityControl" id="opacityControls"><b>Combined opacity:</b>
  <label>Skull <input id="skullOpacity" type="range" min="0.05" max="1" step="0.05" value="0.25"></label>
  <label>Brain <input id="brainOpacity" type="range" min="0.05" max="1" step="0.05" value="0.38"></label>
 </div>
</div>
<div class="grid"><div id="scene3d"></div><div class="right">
 <div class="panel" id="panel0"><div id="view0" style="width:100%;height:100%"></div></div>
 <div class="panel" id="panel1"><div id="view1" style="width:100%;height:100%"></div></div>
 <div class="panel" id="panel2"><div id="view2" style="width:100%;height:100%"></div></div>
</div></div>
<div class="sliders">
 <div class="sliderrow"><b class="sag">Sagittal plane</b><input id="slider0" type="range"><span id="value0"></span></div>
 <div class="sliderrow"><b class="cor">Coronal plane</b><input id="slider1" type="range"><span id="value1"></span></div>
 <div class="sliderrow"><b class="axi">Transverse plane</b><input id="slider2" type="range"><span id="value2"></span></div>
</div></div>
<script>
const packed=__DATA__;
function decode(spec){const raw=atob(spec.b64),bytes=new Uint8Array(raw.length);for(let i=0;i<raw.length;i++)bytes[i]=raw.charCodeAt(i);
 let a;if(spec.dtype.includes('f4'))a=new Float32Array(bytes.buffer);else if(spec.dtype.includes('i4'))a=new Int32Array(bytes.buffer);else a=new Uint8Array(bytes.buffer);
 return {a:a,shape:spec.shape};}
function unpackVolume(p){return {ct:decode(p.ct),brain:decode(p.brain),skull:decode(p.skull),csf:decode(p.csf),steps:p.steps,affine:p.affine};}
function unpackMesh(p){return {v:decode(p.v),f:decode(p.f)};}
const volumes={native:unpackVolume(packed.native),neural:unpackVolume(packed.neural)};
const meshes={brain:unpackMesh(packed.meshes.brain),skull:unpackMesh(packed.meshes.skull),neural:unpackMesh(packed.meshes.neural),csf:unpackMesh(packed.meshes.csf)};
let mode='combined',source=volumes.native;
function columns(a,n){const out=[];for(let c=0;c<n;c++){const q=new Float32Array(a.shape[0]);for(let r=0;r<a.shape[0];r++)q[r]=a.a[r*n+c];out.push(q);}return out;}
function meshTrace(m,name,color,opacity){const nv=m.v.shape[0],nf=m.f.shape[0];return {type:'mesh3d',name:name,color:color,opacity:opacity,
 x:columns(m.v,3)[0],y:columns(m.v,3)[1],z:columns(m.v,3)[2],i:columns(m.f,3)[0],j:columns(m.f,3)[1],k:columns(m.f,3)[2],
 flatshading:false,lighting:{ambient:.55,diffuse:.8,specular:.18,roughness:.65},hoverinfo:'name'};}
function world(v,i,j,k){const A=v.affine,s=v.steps,x=i*s[0],y=j*s[1],z=k*s[2];return [A[0][0]*x+A[0][1]*y+A[0][2]*z+A[0][3],A[1][0]*x+A[1][1]*y+A[1][2]*z+A[1][3],A[2][0]*x+A[2][1]*y+A[2][2]*z+A[2][3]];}
function flat(v,i,j,k){const sh=v.ct.shape;return (i*sh[1]+j)*sh[2]+k;}
function slice2d(v,field,axis,index){const sh=v.ct.shape,a=v[field].a;let rows,cols,out;
 if(axis===0){rows=sh[2];cols=sh[1];out=new Array(rows);for(let r=0;r<rows;r++){out[r]=new Array(cols);for(let c=0;c<cols;c++)out[r][c]=a[flat(v,index,c,sh[2]-1-r)];}}
 else if(axis===1){rows=sh[2];cols=sh[0];out=new Array(rows);for(let r=0;r<rows;r++){out[r]=new Array(cols);for(let c=0;c<cols;c++)out[r][c]=a[flat(v,c,index,sh[2]-1-r)];}}
 else{rows=sh[1];cols=sh[0];out=new Array(rows);for(let r=0;r<rows;r++){out[r]=new Array(cols);for(let c=0;c<cols;c++)out[r][c]=a[flat(v,c,sh[1]-1-r,index)];}}return out;}
function maskedSlice(v,field,axis,index){const z=slice2d(v,field,axis,index);for(let r=0;r<z.length;r++)for(let c=0;c<z[r].length;c++)z[r][c]=z[r][c]?1:null;return z;}
function extents(v){const sh=v.ct.shape;return [[0,sh[0]-1],[0,sh[1]-1],[0,sh[2]-1]];}
function glassCoords(v,axis,index){const e=extents(v);let q;if(axis===0)q=[[index,e[1][0],e[2][0]],[index,e[1][1],e[2][0]],[index,e[1][1],e[2][1]],[index,e[1][0],e[2][1]]];
 else if(axis===1)q=[[e[0][0],index,e[2][0]],[e[0][1],index,e[2][0]],[e[0][1],index,e[2][1]],[e[0][0],index,e[2][1]]];
 else q=[[e[0][0],e[1][0],index],[e[0][1],e[1][0],index],[e[0][1],e[1][1],index],[e[0][0],e[1][1],index]];return q.map(p=>world(v,p[0],p[1],p[2]));}
const colors=['#57d7ff','#ff68cf','#71e875'];
function glassTrace(axis,v,index){const p=glassCoords(v,axis,index);return {type:'mesh3d',name:['Sagittal glass plane','Coronal glass plane','Transverse glass plane'][axis],
 x:p.map(q=>q[0]),y:p.map(q=>q[1]),z:p.map(q=>q[2]),i:[0,0],j:[1,2],k:[2,3],color:colors[axis],opacity:.13,hoverinfo:'name',showlegend:true};}
function lineTrace(axis,v,index){const p=glassCoords(v,axis,index),q=[p[0],p[1],p[2],p[3],p[0]];return {type:'scatter3d',mode:'lines',name:'plane outline',showlegend:false,
 x:q.map(x=>x[0]),y:q.map(x=>x[1]),z:q.map(x=>x[2]),line:{color:colors[axis],width:4},hoverinfo:'skip'};}
const initial=[Math.floor(source.ct.shape[0]/2),Math.floor(source.ct.shape[1]/2),Math.floor(source.ct.shape[2]/2)];
const sceneData=[meshTrace(meshes.skull,'Skull','#ead8ad',.23),meshTrace(meshes.brain,'Brain','#ef9187',.38),meshTrace(meshes.neural,'SIREN neural brain','#ff9c8f',.38),
 meshTrace(meshes.csf,'Clean native-CT CSF-like surface','#168bff',.45)];
for(let a=0;a<3;a++){sceneData.push(glassTrace(a,source,initial[a]));sceneData.push(lineTrace(a,source,initial[a]));}
const sceneLayout={paper_bgcolor:'#000',plot_bgcolor:'#000',font:{color:'#eee'},margin:{l:0,r:0,t:45,b:0},title:{text:'3D skull + brain with transparent plane guides',font:{size:16,color:'#eee'}},
 scene:{bgcolor:'#000',aspectmode:'data',xaxis:{title:'Patient X (mm)',gridcolor:'#303030',zerolinecolor:'#555'},yaxis:{title:'Patient Y (mm)',gridcolor:'#303030',zerolinecolor:'#555'},zaxis:{title:'Patient Z (mm)',gridcolor:'#303030',zerolinecolor:'#555'},camera:{eye:{x:1.5,y:-1.7,z:1.2}}},
 legend:{x:.01,y:.99,bgcolor:'rgba(0,0,0,.55)',font:{color:'#ddd'}}};
Plotly.newPlot('scene3d',sceneData,sceneLayout,{responsive:true,displaylogo:false});
function contourTrace(z,color,name,width){return {type:'contour',z:z,showscale:false,hoverinfo:'skip',connectgaps:false,opacity:1,
 contours:{start:.5,end:.5,size:1,coloring:'lines',showlabels:false},line:{color:color,width:width},name:name};}
function panelData(axis,index){return [
 {type:'heatmap',z:slice2d(source,'ct',axis,index),colorscale:[[0,'#000000'],[1,'#ffffff']],zmin:-30,zmax:180,zsmooth:false,showscale:false,hovertemplate:'HU=%{z:.1f}<extra></extra>'},
 contourTrace(maskedSlice(source,'skull',axis,index),'#ffd21f','Skull contour',1.15),
 contourTrace(maskedSlice(source,'brain',axis,index),'#ff5a4f','Brain contour',1.05),
 contourTrace(maskedSlice(source,'csf',axis,index),'#168bff','CSF-like contour',1.25)];}
function panelLayout(axis){return {paper_bgcolor:'#000',plot_bgcolor:'#000',font:{color:'#eee'},margin:{l:2,r:2,t:31,b:2},title:{text:['Sagittal CT output','Coronal CT output','Transverse / axial CT output'][axis],font:{size:14,color:'#eee'}},
 xaxis:{visible:false,constrain:'domain'},yaxis:{visible:false,scaleanchor:'x'},showlegend:false};}
for(let a=0;a<3;a++)Plotly.newPlot('view'+a,panelData(a,initial[a]),panelLayout(a),{responsive:true,displaylogo:false});
function setSliderRanges(reset){for(let a=0;a<3;a++){const s=document.getElementById('slider'+a);s.min=0;s.max=source.ct.shape[a]-1;s.step=1;if(reset)s.value=Math.floor((source.ct.shape[a]-1)/2);document.getElementById('value'+a).textContent='voxel '+(s.value*source.steps[a]);}}
function windowFor(){if(mode==='skull')return[-800,2000];if(mode==='combined')return[-30,180];return[0,80];}
function updatePlane(axis){const s=document.getElementById('slider'+axis),idx=Number(s.value);document.getElementById('value'+axis).textContent='voxel '+(idx*source.steps[axis]);
 const g=glassTrace(axis,source,idx),l=lineTrace(axis,source,idx),gi=4+2*axis,li=gi+1;
 Plotly.restyle('scene3d',{x:[g.x],y:[g.y],z:[g.z]},[gi]);Plotly.restyle('scene3d',{x:[l.x],y:[l.y],z:[l.z]},[li]);
 const w=windowFor();Plotly.restyle('view'+axis,{z:[slice2d(source,'ct',axis,idx),maskedSlice(source,'skull',axis,idx),maskedSlice(source,'brain',axis,idx),maskedSlice(source,'csf',axis,idx)]},[0,1,2,3]);
 Plotly.restyle('view'+axis,{zmin:w[0],zmax:w[1]},[0]);}
function applyOpacity(){if(mode!=='combined')return;Plotly.restyle('scene3d',{opacity:Number(document.getElementById('skullOpacity').value)},[0]);Plotly.restyle('scene3d',{opacity:Number(document.getElementById('brainOpacity').value)},[1]);}
function applyMode(newMode){mode=newMode;source=(mode==='neural')?volumes.neural:volumes.native;setSliderRanges(true);
 const showSkull=mode==='skull'||mode==='combined',showBrain=mode==='brain'||mode==='combined',showNeural=mode==='neural';
 document.getElementById('opacityControls').style.display=(mode==='combined')?'flex':'none';
 Plotly.restyle('scene3d',{visible:showSkull,opacity:mode==='skull'?.46:.25},[0]);Plotly.restyle('scene3d',{visible:showBrain,opacity:mode==='brain'?.46:.38},[1]);Plotly.restyle('scene3d',{visible:showNeural,opacity:.44},[2]);
 if(mode==='combined')applyOpacity();
 const title=mode==='skull'?'3D skull':mode==='brain'?'3D brain':mode==='neural'?'3D SIREN neural CT brain':'3D skull + brain';Plotly.relayout('scene3d',{'title.text':title+' with transparent glass plane guides'});
 for(let a=0;a<3;a++){Plotly.restyle('view'+a,{visible:showSkull},[1]);Plotly.restyle('view'+a,{visible:showBrain||showNeural},[2]);updatePlane(a);}applyCsf();}
function applyCsf(){const on=document.getElementById('showCsf').checked,color=document.getElementById('csfColor').value,opacity=Number(document.getElementById('csfOpacity').value);
 Plotly.restyle('scene3d',{visible:on,color:color,opacity:opacity},[3]);for(let a=0;a<3;a++)Plotly.restyle('view'+a,{visible:on,'line.color':color},[3]);}
function applyPlaneVisibility(axis){const on=document.getElementById(['showSag','showCor','showAxi'][axis]).checked;document.getElementById('panel'+axis).classList.toggle('hidden',!on);
 Plotly.restyle('scene3d',{visible:on},[4+2*axis,5+2*axis]);setTimeout(()=>{Plotly.Plots.resize('scene3d');for(let a=0;a<3;a++)if(!document.getElementById('panel'+a).classList.contains('hidden'))Plotly.Plots.resize('view'+a);},60);}
document.querySelectorAll('input[name=model]').forEach(r=>r.addEventListener('change',e=>applyMode(e.target.value)));
for(let a=0;a<3;a++){document.getElementById('slider'+a).addEventListener('input',()=>updatePlane(a));document.getElementById(['showSag','showCor','showAxi'][a]).addEventListener('change',()=>applyPlaneVisibility(a));}
document.getElementById('showCsf').addEventListener('change',applyCsf);document.getElementById('csfColor').addEventListener('input',applyCsf);document.getElementById('csfOpacity').addEventListener('input',applyCsf);document.getElementById('skullOpacity').addEventListener('input',applyOpacity);document.getElementById('brainOpacity').addEventListener('input',applyOpacity);
document.querySelector('input[name=model][value=combined]').checked=true;setSliderRanges(true);applyMode('combined');
</script></body></html>'''

dashboard_html=HTML_TEMPLATE.replace("__PLOTLY__",PLOTLY_JS).replace("__DATA__",DATA_JSON)
dashboard_path=OUTPUT_DIR/"CASE_001_manual_interactive_3d_multiplanar_dashboard.html"
dashboard_path.write_text(dashboard_html,encoding="utf-8")
display(HTML(dashboard_html))
print("Saved single dashboard:",dashboard_path)

Saved single dashboard: F:\csstdy\outputs\14_interactive_multiplanar_3d\CASE_001_manual_interactive_3d_multiplanar_dashboard.html


## 4. Reproducibility summary

In [5]:
summary={
    "case":"CASE_001",
    "dashboard":str(dashboard_path),
    "layout":"one 3D model left plus three separate 2D plane outputs right",
    "manual_controls_only":True,
    "model_options":["Skull","Brain","Skull + Brain","Neural CT Brain"],
    "plane_options":["Sagittal","Coronal","Transverse / Axial"],
    "plane_selection":"any one, any two, or all three using checkboxes",
    "3d_plane_style":"transparent glass guide without CT texture",
    "background":"explicit black for page, 3D scene and 2D CT panels",
    "mask_display":"thin skull/brain/native-CSF contours; CSF off by default",
    "csf_3d_display":"cleaned and Gaussian-smoothed native-CT surface; quantitative native mask unchanged",
    "csf_controls":["visibility","color","opacity"],
    "combined_opacity_controls":"separate skull and brain sliders visible only in Skull + Brain mode",
    "background":"explicit black for page, 3D scene and 2D CT panels",
    "mask_display":"thin skull/brain/native-CSF contours; CSF off by default",
    "csf_3d_display":"cleaned and Gaussian-smoothed native-CT surface; quantitative native mask unchanged",
    "csf_controls":["visibility","color","opacity"],
    "combined_opacity_controls":"separate skull and brain sliders visible only in Skull + Brain mode",
    "2d_output":"CT image displayed separately beside the 3D model",
    "csf_overlay":{
        "color":"blue","source":"original CT","hu_range":[0,20],"brain_erosion_iterations":3,
        "minimum_component_volume_mm3":10.0,"reference_match_notebook_11":reference_match,
        "measurement_mask":str(csf_path),"clean_display_mask":str(csf_clean_path),"interpretation":"CSF-like low-HU candidate; not diagnostic"
    },
    "neural_provenance":"anonymized DICOM -> NIfTI CT -> SIREN neural HU reconstruction",
    "limitations":[
        "Blue low-HU candidates are not a dedicated ventricular segmentation.",
        "Automated brain and skull masks may contain errors.",
        "The SIREN neural reconstruction is not clinical ground truth.",
        "Research visualization only; radiological interpretation requires expert review."
    ]
}
(OUTPUT_DIR/"manual_dashboard_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print("Manual interactive dashboard completed.")
print("Notebook-11 CSF count cross-check:",reference_match)
print("Open for full-screen use:",dashboard_path)

Manual interactive dashboard completed.
Notebook-11 CSF count cross-check: True
Open for full-screen use: F:\csstdy\outputs\14_interactive_multiplanar_3d\CASE_001_manual_interactive_3d_multiplanar_dashboard.html
